In [2]:
import requests
import time
import csv

In [3]:
years = list(range(2010, 2026))  
demographics = ['shounen', 'shoujo', 'seinen', 'josei']
batch_size = 100
max_offset = 10000 
sleep_time = 2  


output_file = 'mangadex_data.csv'

header = ['id', 'title', 'description', 'status', 'originalLanguage',
          'publicationDemographic', 'year', 'lastChapter']

total_entries_collected = 0

with open(output_file, mode='w', encoding='utf-8', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(header)

    for year in years:
        for demo in demographics:
            offset = 0
            while offset < max_offset:
                print(f"Fetching for year {year}, demographic {demo}, offset {offset}...")
                params = {
                    "limit": batch_size,
                    "offset": offset,
                    "publicationDemographic[]": demo,
                    "originalLanguage[]": "ja",
                    "order[year]": "desc",
                    "year": year,
                    "includes[]": ["statistics"]
                }

                try:
                    response = requests.get("https://api.mangadex.org/manga", params=params)
                    if response.status_code == 200:
                        json_data = response.json()
                        manga_list = json_data.get("data", [])
                        if not manga_list:
                            print(f"No data at offset {offset}, stopping this batch.")
                            break

                        for manga in manga_list:
                            attributes = manga.get("attributes", {})
                            stats = manga.get("statistics", {}).get(manga["id"], {})
                            title = attributes.get("title", {}).get("en", "")
                            description = attributes.get("description", {}).get("en", "")
                            row = [
                                manga["id"],
                                title,
                                description,
                                attributes.get("status", ""),
                                attributes.get("originalLanguage", ""),
                                attributes.get("publicationDemographic", ""),
                                attributes.get("year", ""),
                                attributes.get("lastChapter", ""),
                                stats.get("rating", {}).get("average", "")
                            ]
                            writer.writerow(row)
                            total_entries_collected += 1

                        offset += batch_size
                        time.sleep(sleep_time)
                    else:
                        print(f"Failed at offset {offset} (year={year}, demo={demo}): Status {response.status_code}")
                        break

                except Exception as e:
                    print(f"Error at offset {offset} (year={year}, demo={demo}): {e}")
                    break

estimated_pages = total_entries_collected // 4  
print(f"\nCollected {total_entries_collected} manga entries in total.")
print(f"Estimated pages: {estimated_pages}")


Fetching for year 2010, demographic shounen, offset 0...
Fetching for year 2010, demographic shounen, offset 100...
Fetching for year 2010, demographic shounen, offset 200...
Fetching for year 2010, demographic shounen, offset 300...
🟡 No data at offset 300, stopping this batch.
Fetching for year 2010, demographic shoujo, offset 0...
Fetching for year 2010, demographic shoujo, offset 100...
Fetching for year 2010, demographic shoujo, offset 200...
🟡 No data at offset 200, stopping this batch.
Fetching for year 2010, demographic seinen, offset 0...
Fetching for year 2010, demographic seinen, offset 100...
Fetching for year 2010, demographic seinen, offset 200...
Fetching for year 2010, demographic seinen, offset 300...
Fetching for year 2010, demographic seinen, offset 400...
🟡 No data at offset 400, stopping this batch.
Fetching for year 2010, demographic josei, offset 0...
Fetching for year 2010, demographic josei, offset 100...
🟡 No data at offset 100, stopping this batch.
Fetching f